NOTEBOOK 3 : JOINTURES ET AGREGATIONS

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TradeCorp_Nettoyage") \
    .getOrCreate()

df_customers = spark.read.parquet("../data/tmp/customers")
df_orders= spark.read.parquet("../data/tmp/orders")
df_order_details= spark.read.parquet("../data/tmp/order_details")
df_products= spark.read.parquet("../data/tmp/products")
df_employees= spark.read.parquet("../data/tmp/employees")
df_categories = spark.read.csv("/home/jovyan/data/categories.csv", header=True, inferSchema=True)
df_suppliers = spark.read.csv("/home/jovyan/data/suppliers.csv", header=True, inferSchema=True)
df_shippers = spark.read.csv("/home/jovyan/data/shippers.csv", header=True, inferSchema=True)

print("---Spark est activé et toutes les tables sont chargées---")

---Spark est activé et toutes les tables sont chargées---


Q21—Jointure orders + customers

In [3]:
# Joindre df_orders et df_customers sur customer_id
df_jointure = df_orders.join(df_customers,"customer_id","inner") \
         .select("order_id","company_name","country", "order_date","freight")

# résultat 
df_jointure.show(5)

+--------+--------------------+-------+----------+-------+
|order_id|        company_name|country|order_date|freight|
+--------+--------------------+-------+----------+-------+
|   10400|  Eastern Connection|     UK|1997-01-01|  83.93|
|   10401|Rattlesnake Canyo...|    USA|1997-01-01|  12.51|
|   10402|        Ernst Handel|AUSTRIA|1997-01-02|  67.88|
|   10403|        Ernst Handel|AUSTRIA|1997-01-03|  73.79|
|   10404|Magazzini Aliment...|  ITALY|1997-01-03| 155.97|
+--------+--------------------+-------+----------+-------+
only showing top 5 rows


Q22—Jointure order_details + products

In [4]:
# jointure de df_order_details et df_products sur product_id
df_jointure1 = df_order_details.join(df_products,"product_id","inner") \
            .select( df_order_details ["*"], 
                     df_products["product_name"],
                     df_products["category_id"],
                     df_products["unit_price"])

# résultat
df_jointure1.show(5)


+--------+----------+-------------+--------+--------+----------+--------------------+-----------+----------+
|order_id|product_id|prix_unitaire|quantite|discount|sous_total|        product_name|category_id|unit_price|
+--------+----------+-------------+--------+--------+----------+--------------------+-----------+----------+
|   10248|        11|         14.0|      12|     0.0|     168.0|      Queso Cabrales|          4|      21.0|
|   10248|        72|         34.8|       5|     0.0|     174.0|Mozzarella di Gio...|          4|      34.8|
|   10249|        14|         18.6|       9|     0.0|     167.4|                Tofu|          7|     23.25|
|   10249|        51|         42.4|      40|     0.0|    1696.0|Manjimup Dried Ap...|          7|      53.0|
|   10250|        41|          7.7|      10|     0.0|      77.0|Jack's New Englan...|          8|      9.65|
+--------+----------+-------------+--------+--------+----------+--------------------+-----------+----------+
only showing top 5 

Q23—Jointure products + categories

In [5]:
# Joindre df_products et df_categories sur category_id
df_jointures = df_products.join(df_categories, "category_id","inner") \
            .select(df_products ["*"],
                    df_categories["category_name"],
                    df_categories["description"] )
# résultat 
df_jointures.show(5)

+----------+--------------------+-----------+-----------+-------------------+----------+--------------+--------------+-------------+------------+--------+-------------+--------------------+
|product_id|        product_name|supplier_id|category_id|  quantity_per_unit|unit_price|units_in_stock|units_on_order|reorder_level|discontinued|en_stock|category_name|         description|
+----------+--------------------+-----------+-----------+-------------------+----------+--------------+--------------+-------------+------------+--------+-------------+--------------------+
|         3|       Aniseed Syrup|          1|          2|12 - 550 ml bottles|      10.0|            13|            70|           25|           0|    true|   Condiments|Sweet and savory ...|
|         4|Chef Anton's Caju...|          2|          2|     48 - 6 oz jars|      22.0|            53|             0|            0|           0|    true|   Condiments|Sweet and savory ...|
|         6|Grandma's Boysenb...|          3|     

Q24—DataFrame enrichi complet

In [6]:
from collections import Counter  

# A ) réalisation d'une première jointure
df_jointures4 = df_order_details \
              .join(df_orders, "order_id","inner")\
              .join(df_customers, "customer_id","inner")\
              .join(df_jointures, "product_id","inner")\
              .join(df_employees, "employee_id","inner")\
              .join(df_shippers, "shipper_id","inner")

# utilisation de counter
colonnes_doubles = df_jointures4.columns
nb_col = Counter(colonnes_doubles)

# on filtre les colonnes en doubles 
colonnes_en_double = [col for col, count in nb_col.items() if count > 1]

# résultat
print("les colonnes en doubles sont :", colonnes_en_double )



les colonnes en doubles sont : ['company_name', 'city', 'country', 'phone']


Q24 B) 

In [7]:
from collections import Counter  

# renommage des colonnes 
df_customers_renamed = (df_customers
                        .withColumnRenamed("company_name","customer_company_name")
                        .withColumnRenamed("country","customer_country")
                        .withColumnRenamed("city","customer_city")
                        .withColumnRenamed("phone","customer_phone")
                       )

df_employees_renamed = (df_employees
                        .withColumnRenamed("country","employee_country")
                        .withColumnRenamed("city","employee_city")  
                       )

df_shippers_renamed = (df_shippers
                        .withColumnRenamed("company_name","shipper_company_name")
                        .withColumnRenamed("phone","shipper_phone")
                      )

# création du DF df_orders_enriched avec ces tables renommées
df_orders_enriched = (df_order_details 
              .join(df_orders, "order_id","inner")
              .join(df_customers_renamed, "customer_id","inner")
              .join(df_jointures, "product_id","inner")
              .join(df_employees_renamed, "employee_id","inner")
              .join(df_shippers_renamed, "shipper_id","inner")
                     )

# résultat pour visuel des doublons restants 
nvel_col = df_orders_enriched.columns
nb_cols = Counter(nvel_col)

# on filtre les colonnes en doubles 
colonnes_en_doubles = [col for col, count in nb_cols.items() if count > 1]

# résultat
print("les colonnes en doubles sont :", colonnes_en_doubles )

les colonnes en doubles sont : []


Q25—CA par client

In [9]:
from pyspark.sql.functions import desc,round,col
# Calcul du chiffre d'affaires total par client(company_name)depuis df_orders_enriched
df_ca_clients = (df_orders_enriched 
                 .groupBy("customer_company_name")
                 .sum("sous_total")
                 .withColumn("sum(sous_total)", round(col("sum(sous_total)"), 2))
                 .orderBy(desc("sum(sous_total)")
                ))

# Résultat 
df_ca_clients.show(10)                 

+---------------------+---------------+
|customer_company_name|sum(sous_total)|
+---------------------+---------------+
|           QUICK-Stop|       51682.74|
|   Save-a-lot Markets|       40238.09|
|         Ernst Handel|       39975.91|
|       Mère Paillarde|       22871.07|
| Rattlesnake Canyo...|        17636.1|
|        Simons bistro|       16232.42|
| Hungry Owl All-Ni...|       14403.03|
|       Folk och fä HB|       13200.92|
|     HILARION-Abastos|       11799.74|
|   Berglunds snabbköp|       11758.92|
+---------------------+---------------+
only showing top 10 rows


Q26—CA par catégorie

In [10]:
from pyspark.sql.functions import desc,round,countDistinct,sum
# Calcul du chiffre d'affaires total par catégorie de produits
df_ca_categorie = (df_orders_enriched 
                 .groupBy("category_name")
                 .agg (round(sum("sous_total"),2).alias("ca_total"),
                  countDistinct("product_id").alias("nb_produits_distincts")) 
                 .orderBy(desc("ca_total"))
                )
# Résultat 
df_ca_categorie.show()   

+--------------+--------+---------------------+
| category_name|ca_total|nb_produits_distincts|
+--------------+--------+---------------------+
|Dairy Products|108086.9|                    9|
|     Beverages|90368.64|                    9|
|   Confections|82657.78|                   13|
|       Seafood|66959.23|                   12|
|    Condiments| 54995.0|                   11|
|Grains/Cereals|51463.63|                    6|
|       Produce|40992.09|                    4|
|  Meat/Poultry|11017.17|                    2|
+--------------+--------+---------------------+



Q27—CA par mois

In [11]:
from pyspark.sql.functions import round,sum,date_trunc,col,desc
# Calcul du chiffre d'affaires mensuel
df_ca_mensuel = (df_orders_enriched 
                 .groupBy(date_trunc("month",col("order_date")).alias ("mois_annee"))
                 .agg (round(sum("sous_total"),2).alias("ca_total"))
                 .orderBy(desc("mois_annee"))
                )
# Résultat 
df_ca_mensuel.show()   

+-------------------+--------+
|         mois_annee|ca_total|
+-------------------+--------+
|1997-12-01 00:00:00| 54984.7|
|1997-11-01 00:00:00|39898.78|
|1997-10-01 00:00:00| 48574.5|
|1997-09-01 00:00:00|43335.43|
|1997-08-01 00:00:00|38039.93|
|1997-07-01 00:00:00|45162.88|
|1997-06-01 00:00:00|29875.47|
|1997-05-01 00:00:00|48895.27|
|1997-04-01 00:00:00| 41510.6|
|1997-03-01 00:00:00|33226.33|
|1997-02-01 00:00:00|31549.04|
|1997-01-01 00:00:00|51487.51|
+-------------------+--------+



Q28—Performance par employé

In [12]:
from pyspark.sql import functions as F
# calcul des performances des employés
df_perf_employes = (df_orders_enriched 
                 .groupBy("full_name")
                 .agg (
                  F.countDistinct("order_id").alias("nb_commandes"), 
                  F.round(F.sum("sous_total"),2).alias("ca_total"),
                  F.round(F.avg(F.datediff(F.col("shipped_date"), F.col("order_date"))), 1).alias("delai_moyen_livraison")
                   )
                 .orderBy(F.desc("ca_total"))
                )
# Résultat 
df_perf_employes.show()   

+----------------+------------+---------+---------------------+
|       full_name|nb_commandes| ca_total|delai_moyen_livraison|
+----------------+------------+---------+---------------------+
|Margaret Peacock|          75|104193.78|                  8.3|
| Janet Leverling|          71| 97081.27|                  8.9|
|   Nancy Davolio|          54| 81898.92|                  7.8|
|   Andrew Fuller|          40| 54907.03|                 10.2|
|     Robert King|          33| 49562.78|                  9.8|
|  Laura Callahan|          53| 47077.95|                  8.0|
|  Michael Suyama|          33| 34037.07|                  7.9|
|  Anne Dodsworth|          18| 20595.99|                 10.0|
| Steven Buchanan|          18| 17185.65|                  6.5|
+----------------+------------+---------+---------------------+



Q29—Window functions—Rang

In [16]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# partitionnement par catégorie et tri par ordre décroissant 
window_spec = Window.partitionBy("category_name").orderBy(F.desc("ca"))

# calcul du CA total pour chaque produit 
df_ca_produits = (df_orders_enriched 
                 .groupBy("category_name","product_name")
                 .agg (round(F.sum("sous_total"),2).alias("ca"))
                )

# application de la fonction de classement avec dense_rank
df_classement = (df_ca_produits
    .withColumn("rang", F.dense_rank().over(window_spec))
    .orderBy("category_name", "rang") # Tri purement visuel pour l'affichage
)
# Résultat 
df_classement.show(10)   

+-------------+--------------------+--------+----+
|category_name|        product_name|      ca|rang|
+-------------+--------------------+--------+----+
|    Beverages|       Côte de Blaye|49198.09|   1|
|    Beverages|         Ipoh Coffee| 11069.9|   2|
|    Beverages|        Lakkalikööri|  7379.1|   3|
|    Beverages|       Outback Lager|  5468.4|   4|
|    Beverages|      Steeleye Stout|  5274.9|   5|
|    Beverages|Rhönbräu Klosterbier| 4485.55|   6|
|    Beverages|    Chartreuse verte|  4475.7|   7|
|    Beverages|       Sasquatch Ale|  2107.0|   8|
|    Beverages|Laughing Lumberja...|   910.0|   9|
|   Condiments|Louisiana Fiery H...|  9373.2|   1|
+-------------+--------------------+--------+----+
only showing top 10 rows


Q30—Window functions—Cumul

In [15]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# définition de la fenêtre 
window_spec = Window.orderBy("mois_annee").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# calcul du CA cumulé
df_ca_cumule = (df_orders_enriched 
                 .groupBy(F.date_trunc("month",F.col("order_date")).alias ("mois_annee"))
                 .agg (F.round(F.sum("sous_total"),2).alias("ca_mensuel"))
                 .withColumn("ca_cumule", F.round(F.sum("ca_mensuel").over(window_spec), 2))
                 .orderBy(F.desc("mois_annee"))
                )

# résultat
df_ca_cumule.show(15)

+-------------------+----------+---------+
|         mois_annee|ca_mensuel|ca_cumule|
+-------------------+----------+---------+
|1997-12-01 00:00:00|   54984.7|506540.44|
|1997-11-01 00:00:00|  39898.78|451555.74|
|1997-10-01 00:00:00|   48574.5|411656.96|
|1997-09-01 00:00:00|  43335.43|363082.46|
|1997-08-01 00:00:00|  38039.93|319747.03|
|1997-07-01 00:00:00|  45162.88| 281707.1|
|1997-06-01 00:00:00|  29875.47|236544.22|
|1997-05-01 00:00:00|  48895.27|206668.75|
|1997-04-01 00:00:00|   41510.6|157773.48|
|1997-03-01 00:00:00|  33226.33|116262.88|
|1997-02-01 00:00:00|  31549.04| 83036.55|
|1997-01-01 00:00:00|  51487.51| 51487.51|
+-------------------+----------+---------+



Q31—Tri et limite

In [18]:
from pyspark.sql import functions as F

# calcul du CA par pays et par produit 
df_ca_pays_produit = (df_orders_enriched 
                 .groupBy("ship_country","product_name")
                 .agg (F.round(F.sum("sous_total"),2).alias("ca_total"))
                 .orderBy(F.desc("ca_total")) 
                 .limit(5)
                )

# calcul des 3 pays clients qui génèrent le plus de CA
df_pays_client = (df_orders_enriched 
                 .groupBy("customer_country")
                 .agg (F.round(F.sum("sous_total"),2).alias("ca_total"))
                 .orderBy(F.desc("ca_total")) 
                 .limit(3)
                )

# résultats
df_ca_pays_produit.show()
df_pays_client.show()




+------------+--------------------+--------+
|ship_country|        product_name|ca_total|
+------------+--------------------+--------+
|         USA|       Côte de Blaye|12713.88|
|     Denmark|       Côte de Blaye| 10540.0|
|     Germany|Raclette Courdavault| 10164.0|
|      Canada|       Côte de Blaye| 8263.36|
|     Germany|       Côte de Blaye|  7905.0|
+------------+--------------------+--------+

+----------------+---------+
|customer_country| ca_total|
+----------------+---------+
|         GERMANY|100641.29|
|             USA| 90731.71|
|         AUSTRIA| 46559.49|
+----------------+---------+



Q32 — Écriture en Parquet

In [20]:
# écriture du fichier df_orders_enriched au format parquet 
df_orders_enriched.write.mode("overwrite").parquet("/home/jovyan/data/output/orders_enriched.parquet")

print("Fichier sauvegardé !")


Fichier sauvegardé !
